# Estimation R and wave direction
Rを求めて海表面情報を探索する試み。

In [2]:
# 依存パッケージをimport

import os

from gnssrefl.utils import check_environment, set_environment, get_sys
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import pywt

# import gnssrefl functions
from gnssrefl.rinex2snr_cl import rinex2snr
from gnssrefl.gnssir_cl import gnssir

#@formatter:off
%matplotlib inline

## RINEX → SNR

下記ディレクトリにRINEXファイルを入れる。

refl_code/yyyy/rinex/TTTT/TTTTXXX0.yyo
- yyyy：西暦
- TTTT：局名。アルファベット4文字。
- XXX：day of year
- yy：西暦下二桁

In [9]:
year = 2020
doy = 277
doy_end = 288
station = "calc"

# rinex2snr(station, year, doy, doy_end=doy_end, snr=88, nolook=True)
rinex2snr("uuav", 2024, 345, snr=88, nolook=True)

make output directory for snr files in  2024
No json file found - but you have requested the code not exit
Using snr value of  88
Using default orbit for this time period:  rapid
Using archive value of  all
Station  uuav  has four characters, assume RINEX 2.11 format
Debug mode or only analyzing one day of data. 
SNR file does not already exist. Which means I will try to make it.
uuav 2024 345
Fortran translation log:  /etc/gnssrefl/refl_code/logs/uuav/2024/345_translation.txt
General log:  /etc/gnssrefl/refl_code/logs/uuav/2024/345_translation.txt.gen
Will first assume station  uuav  year: 2024  doy: 345 is located here : /usr/src/gnssrefl/notebooks/Dev/Wind_Direction/EstimateR
Looked for  /usr/src/gnssrefl/notebooks/Dev/Wind_Direction/EstimateR/uuav3450.24o
Failed, will search other names/directories
Looking for file:  /usr/src/gnssrefl/notebooks/Dev/Wind_Direction/EstimateR/uuav3450.24o.gz
Looking for file:  /usr/src/gnssrefl/notebooks/Dev/Wind_Direction/EstimateR/uuav3450.24o.Z
Loo

## SNRファイルの読み込み関数

In [12]:
from pathlib import Path
import pandas as pd
import numpy as np

# SNR ファイルのカラム名（マニュアル準拠）
SNR_COLUMNS_FULL = [
    "sat",              # 衛星番号
    "elev_deg",         # 仰角 [deg]
    "az_deg",           # 方位角 [deg]
    "sec_of_day",       # 秒（GPS time）
    "edot_deg_per_s",   # 仰角変化率 [deg/s]
    "snr_L6",           # S6 (L6) SNR [dB-Hz]
    "snr_L1",           # S1 (L1)
    "snr_L2",           # S2 (L2)
    "snr_L5",           # S5 (L5)
    "snr_L7",           # S7 (L7)
    "snr_L8",           # S8 (L8)
]


def read_snr_file_gz(path: Path) -> pd.DataFrame:
    """1つの .snr*.gz ファイルを読み込んで DataFrame を返す。"""
    df = pd.read_csv(
        path,
        sep=r"\s+",        # ← FutureWarning 回避
        header=None,
        comment="#",
        compression="gzip",
        engine="python",
    )
    n_cols = df.shape[1]
    df.columns = SNR_COLUMNS_FULL[:n_cols]
    return df


def load_snr_directory(year: int, station: str,
                       base_dir: str = "/etc/gnssrefl/refl_code"):
    """
    /etc/gnssrefl/refl_code/{year}/snr/{station}/ 配下の
    .snr*.gz ファイルだけを全部読み込んで

    - snr_dict: {ファイル名: DataFrame}
    - snr_table: 全ファイルを縦結合した DataFrame
                 (year, station, doy, file などの情報付き)

    を返す。
    """
    snr_dir = Path(base_dir) / str(year) / "snr" / station

    if not snr_dir.is_dir():
        raise FileNotFoundError(f"ディレクトリが見つかりません: {snr_dir}")

    snr_files = sorted(snr_dir.glob("*.snr88.gz"))

    if not snr_files:
        print(f"*.snr*.gz が見つかりません: {snr_dir}")

    snr_dict = {}
    table_list = []

    for f in snr_files:
        df = read_snr_file_gz(f)
        snr_dict[f.name] = df

        # ファイル名から DOY を推定（例: calc2770.20.snr66.gz）
        name_no_gz = f.name[:-3] if f.name.endswith(".gz") else f.name
        stem = name_no_gz.split(".")[0]   # "calc2770"
        doy = int(stem[len(station):len(station) + 3])  # "277"

        df2 = df.copy()
        df2["year"] = year
        df2["station"] = station
        df2["file"] = f.name
        df2["doy"] = doy
        table_list.append(df2)

    if table_list:
        snr_table = pd.concat(table_list, ignore_index=True)
    else:
        snr_table = pd.DataFrame()

    return snr_dict, snr_table

def main(year: int,
         station: str,
         snr_dict_existing=None,
         base_dir: str = "/etc/gnssrefl/refl_code"):
    """
    - Jupyter で既に snr_dict を作っているなら snr_dict_existing を渡す
    - そうでなければディレクトリから読み込む

    戻り値:
        snr_dict, snr_table
    """
    if snr_dict_existing is None:
        snr_dict, snr_table = load_snr_directory(year, station, base_dir=base_dir)
    else:
        # 既存の snr_dict から snr_table だけ再構成したい場合など
        snr_dict = snr_dict_existing
        # ここで snr_table を組み立て直すなら、上の build 処理を使い回すイメージ
        # 必要なら別関数に分けてもよい
        rows = []
        for fname, df in snr_dict.items():
            name_no_gz = fname[:-3] if fname.endswith(".gz") else fname
            stem = name_no_gz.split(".")[0]
            doy = int(stem[len(station):len(station) + 3])

            tmp = df.copy()
            tmp["year"] = year
            tmp["station"] = station
            tmp["file"] = fname
            tmp["doy"] = doy
            rows.append(tmp)
        snr_table = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

    return snr_dict, snr_table


## コヒーレント・インコヒーレント比率を求める

In [46]:
LAMBDA_L1 = 0.190293672798365  # [m] GNSS L1 の波長

import matplotlib.pyplot as plt

def plot_R_e(
    e: np.ndarray,
    R: np.ndarray,
    sat: int,
    signal: str,
    doy: int | None = None,
    az_range: tuple[float, float] | None = None,
    direction: str | None = None,
    h0: float | None = None,
    extra_title: str | None = None,
):
    """
    R(e) と R_dB(e) をプロットする。
    衛星・信号などのメタ情報をタイトルに必ず入れる。
    """
    base = f"sat={sat}, sig={signal}"
    if doy is not None:
        base += f", doy={doy}"
    if az_range is not None:
        base += f", az=[{az_range[0]:.1f},{az_range[1]:.1f}]"
    if direction is not None:
        base += f", dir={direction}"
    if h0 is not None:
        base += f", h0={h0:.2f} m"
    if extra_title is not None:
        base += " | " + extra_title

    # 線形スケール
    plt.figure()
    plt.plot(e, R)
    plt.xlabel("Elevation e [deg]")
    plt.ylabel("R(e) = P_inc / P_coh [-]")
    plt.title(base)
    plt.grid(True)
    plt.show()

    # dBスケール
    plt.figure()
    R_dB = 10 * np.log10(R)
    plt.plot(e, R_dB)
    plt.xlabel("Elevation e [deg]")
    plt.ylabel("R(e) [dB]")
    plt.title(base)
    plt.grid(True)
    plt.show()


def plot_wavelet_power_e_h(
    e_grid,
    h_grid,
    P,
    sat: int,
    signal: str,
    doy: int | None = None,
    az_range: tuple[float, float] | None = None,
    direction: str | None = None,
    extra_title: str | None = None,
):
    """
    P(e,h)=|W|^2 をカラーマップで描画。
    """
    import numpy as np
    import matplotlib.pyplot as plt

    e_grid = np.asarray(e_grid)
    h_grid = np.asarray(h_grid)
    P = np.asarray(P, float)

    if P.shape != (len(e_grid), len(h_grid)):
        raise ValueError(
            f"P.shape={P.shape}, expected ({len(e_grid)}, {len(h_grid)})"
        )

    # 0 以上に限定
    P[P < 0] = 0.0
    Pmax = np.nanmax(P)
    if not np.isfinite(Pmax) or Pmax <= 0:
        Pmax = 1.0  # 全部ゼロだったときの保険

    E, H = np.meshgrid(e_grid, h_grid, indexing="xy")

    base = f"sat={sat}, sig={signal}"
    if doy is not None:
        base += f", doy={doy}"
    if az_range is not None:
        base += f", az=[{az_range[0]:.1f},{az_range[1]:.1f}]"
    if direction is not None:
        base += f", dir={direction}"
    if extra_title is not None:
        base += " | " + extra_title

    plt.figure()
    pcm = plt.pcolormesh(E, H, P.T, shading="auto", vmin=0.0, vmax=Pmax)
    plt.xlabel("Elevation e [deg]")
    plt.ylabel("Height h [m]")
    plt.title(base)
    cb = plt.colorbar(pcm)
    cb.set_label(r"$|W(e,h)|^2$")
    plt.tight_layout()
    plt.show()


def detrend_snr_gnss_ir(x, y, poly_order: int = 2):
    coef = np.polyfit(x, y, poly_order)
    trend = np.polyval(coef, x)
    return y - trend, trend


def compute_wavelet_power_e_h(
    elev_deg: np.ndarray,
    snr_series: np.ndarray,
    h_grid: np.ndarray,
    wavelength_m: float = LAMBDA_L1,
    wavelet_name: str = "cmor10.0-5.0",
    nyquist_strict: bool = True,
):
    """
    仰角列 elev_deg と SNR 列 snr_series から 2D パワー P(e,h) を計算する。

    - 横軸は x = sin(e) だが、サンプル位置は「元の仰角の場所だけ」。
    - CWT の sampling_period dt は、以前の実装と同じ
      dt = (x_max - x_min) / (N-1) を使う。
    - h_grid 全体がナイキスト条件を満たさない場合は ValueError。
    """
    elev_deg = np.asarray(elev_deg, float)
    snr_series = np.asarray(snr_series, float)
    h_grid = np.asarray(h_grid, float)

    if elev_deg.size < 4:
        raise ValueError("サンプル数が少なすぎます (N<4)。")

    # 1) e を昇順にソート
    order = np.argsort(elev_deg)
    e_sorted = elev_deg[order]
    y = snr_series[order]

    # 2) x = sin(e)
    x = np.sin(np.deg2rad(e_sorted))

    # ★ 3) 等間隔グリッド x_uniform に補間する
    N = len(x)
    x_min, x_max = x[0], x[-1]
    if N < 2 or x_max <= x_min:
        raise ValueError("sin(e) の範囲が狭すぎます。")

    x_uniform = np.linspace(x_min, x_max, N)
    y_uniform = np.interp(x_uniform, x, y)

    # sampling_period
    dt = x_uniform[1] - x_uniform[0]

    # ★ 4) トレンド除去は x_uniform, y_uniform に対して行う
    y_detr, trend = detrend_snr_gnss_ir(x_uniform, y_uniform, poly_order=2)

    # 5) ナイキストチェック
    h_pos = h_grid[h_grid > 0]
    if h_pos.size == 0:
        raise ValueError("h_grid に正の高さがありません。")
    h_req_max = np.max(h_pos)

    f_x_req_max = 2.0 * h_req_max / wavelength_m
    f_nyq = 1.0 / (2.0 * dt)

    if nyquist_strict and (f_x_req_max > f_nyq):
        h_nyq_max = wavelength_m / (4.0 * dt)
        raise ValueError(
            f"指定された h_max={h_req_max:.3f} m は "
            f"このデータのナイキスト限界 h_max≈{h_nyq_max:.3f} m を超えています。"
        )

    # 6) CWT 準備
    wavelet = pywt.ContinuousWavelet(wavelet_name)
    f_c = pywt.central_frequency(wavelet)

    # 7) h_grid -> scale
    scales = np.zeros_like(h_grid, dtype=float)
    valid_h = h_grid > 0

    f_x = np.zeros_like(h_grid, dtype=float)
    f_x[valid_h] = 2.0 * h_grid[valid_h] / wavelength_m
    scales[valid_h] = f_c / (f_x[valid_h] * dt)  # = λ f_c / (2 h dt)

    valid_idx = np.where(valid_h & np.isfinite(scales) & (scales > 0))[0]
    scales_valid = scales[valid_idx]
    if len(scales_valid) == 0:
        raise ValueError("有効なスケールが 1 つもありません。h_grid の範囲を見直してください。")

    # 8) CWT 実行
    coeffs, _ = pywt.cwt(y_detr, scales_valid, wavelet, sampling_period=dt)
    power_valid = np.abs(coeffs) ** 2  # (n_scales_valid, N)

    # 9) P(e,h) に展開
    P = np.zeros((N, len(h_grid)), dtype=float)
    for j, h_idx in enumerate(valid_idx):
        P[:, h_idx] = power_valid[j, :]

    # 10) e_grid は x_uniform に対応する仰角
    e_grid = np.rad2deg(np.arcsin(x_uniform))

    return e_grid, h_grid, P

def compute_R_from_power(
    e_grid: np.ndarray,
    h_grid: np.ndarray,
    P: np.ndarray,
    h0: float,
    delta_h: float = 4.0,   # ガウス窓モード用
    w_at_delta: float = 0.1,
    sigma_coh: float | None = None,
    use_local_band: bool = False,    # ガウス窓モード用
    h_band_half_width: float = 5.0,  # ガウス窓モード用
    # ★ 追加：高さで明示的にバンドを分けるモード
    coh_h_range: tuple[float, float] | None = None,  # 例: (9.0, 20.0)
    inc_h_range: tuple[float, float] | None = None,  # 例: (0.5, 5.0)
):
    """
    wavelet パワー P(e,h) から、
      - coherent パワー P_coh(e)
      - incoherent パワー P_inc(e)
      - その比 R(e) = P_inc / P_coh
    を求める。

    2通りのモード:

      1) coh_h_range, inc_h_range を指定した場合
         → その高さ帯だけを使って
            P_coh = Σ_{h∈coh} P,  P_inc = Σ_{h∈inc} P

      2) 指定しない場合
         → ガウス窓 w(h) で h0 付近を P_coh とみなし、
            P_total(全h) - P_coh を P_inc とする。
    """
    e_grid = np.asarray(e_grid)
    h_grid = np.asarray(h_grid)
    P = np.asarray(P)

    if P.shape != (len(e_grid), len(h_grid)):
        raise ValueError(
            f"P の shape が (Ne,Nh)={P.shape} ですが、"
            f"Ne={len(e_grid)}, Nh={len(h_grid)} と一致していません。"
        )

    # ------------------------
    # モード1: 明示的な2バンド指定
    # ------------------------
    if (coh_h_range is not None) and (inc_h_range is not None):
        h_coh_min, h_coh_max = coh_h_range
        h_inc_min, h_inc_max = inc_h_range

        # コヒーレント用マスク（例: h >= 9m）
        mask_coh = (h_grid >= h_coh_min) & (h_grid <= h_coh_max)
        # インコヒーレント用マスク（例: h <= 5m）
        mask_inc = (h_grid >= h_inc_min) & (h_grid <= h_inc_max)

        if not np.any(mask_coh):
            raise ValueError("coh_h_range に該当する高さが h_grid 内にありません。")
        if not np.any(mask_inc):
            raise ValueError("inc_h_range に該当する高さが h_grid 内にありません。")

        # 矩形窓で単純和
        P_coh = np.sum(P[:, mask_coh], axis=1)  # (Ne,)
        P_inc = np.sum(P[:, mask_inc], axis=1)  # (Ne,)

        # P_total は「使った2バンドの和」としておく
        P_total = P_coh + P_inc

    else:
        # ------------------------
        # モード2: ガウス窓 w(h) で coherent を抽出
        # ------------------------

        # 1. sigma_coh の決定
        if sigma_coh is None:
            sigma_coh = delta_h / np.sqrt(2.0 * np.log(1.0 / w_at_delta))

        # 2. ガウス窓
        w_coh = np.exp(-(h_grid - h0) ** 2 / (2.0 * sigma_coh ** 2))  # (Nh,)

        if use_local_band:
            mask_band = np.abs(h_grid - h0) <= h_band_half_width
            if not np.any(mask_band):
                raise ValueError("h0±h_band_half_width に該当する高さがありません。")
            w_tmp = np.zeros_like(w_coh)
            w_tmp[mask_band] = w_coh[mask_band]
            w_coh = w_tmp

        # 3. P_total(e) は全高さの和
        P_total = np.sum(P, axis=1)                      # (Ne,)
        #    P_coh(e) はガウス窓重み付き
        P_coh = np.sum(P * w_coh[None, :], axis=1)       # (Ne,)
        P_inc = P_total - P_coh                          # (Ne,)

    # ------------------------
    # R(e) = P_inc / P_coh をそのまま使う
    # ------------------------
    with np.errstate(divide="ignore", invalid="ignore"):
        q_e = np.where(P_coh > 0.0, P_inc / P_coh, np.nan)  # 名称だけ q_e、実体は R
        R_e = q_e

    # e 方向に積分
    P_coh_int = float(np.trapezoid(P_coh, e_grid))
    P_inc_int = float(np.trapezoid(P_inc, e_grid))

    if P_coh_int > 0.0:
        q_int = P_inc_int / P_coh_int
        R_int = q_int
    else:
        q_int = np.nan
        R_int = np.nan

    return {
        "e": e_grid,
        "P_coh": P_coh,
        "P_total": P_total,
        "P_inc": P_inc,
        "sigma_coh": sigma_coh,
        "q": q_e,         # 実質 R(e) = P_inc/P_coh
        "R": R_e,         # 同じもの
        "P_coh_int": P_coh_int,
        "P_inc_int": P_inc_int,
        "q_int": q_int,   # 実質 R_int
        "R_int": R_int,
    }

def save_R_result_resfile(
    filepath: str,
    doy: int,
    sat: int,
    signal: str,
    df_sel: pd.DataFrame,
    res: dict,
):
    """
    R(e) の結果を res ファイルに出力する。
    e, az, R(linear), R(dB) を列として書く。
    """
    e_arr = np.asarray(res["e"])
    R_arr = np.asarray(res["R"])

    # df_sel は select_observations で elev_deg 昇順になっているので、
    # e_arr と同じ順序だと仮定（compute_wavelet_power_e_h でもソート順は保っている）。
    az_arr = df_sel["az_deg"].to_numpy()
    if len(az_arr) != len(e_arr):
        raise ValueError("df_sel と res の長さが一致しません。")

    R_dB = 10.0 * np.log10(R_arr)

    data = np.column_stack([e_arr, az_arr, R_arr, R_dB])

    header = (
        f"# GNSS-IR R(e) result\n"
        f"# doy={doy}, sat={sat}, signal={signal}\n"
        f"# columns: e_deg, az_deg, R_lin, R_dB\n"
    )

    np.savetxt(filepath, data, header=header)


# 信号名→カラム名
SIGNAL_COL_MAP = {
    "L1": "snr_L1",
    "L2": "snr_L2",
    "L5": "snr_L5",
    "L6": "snr_L6",
    "L7": "snr_L7",
    "L8": "snr_L8",
}

SIGNAL_CODE_MAP = {
    "L1": 1,
    "L2": 2,
    "L5": 5,
    "L6": 6,
    "L7": 7,
    "L8": 8,
}


def compute_R_for_single_sat(
    snr_table: pd.DataFrame,
    doy: int,
    sat: int,
    signal: str,
    az_range: tuple[float, float],
    time_range_hour: tuple[float, float],
    h_grid: np.ndarray,
    h0: float,
    wavelength_m: float = LAMBDA_L1,
    wavelet_name: str = "cmor5.0-1.0",
    nyquist_strict: bool = True,
    min_points: int = 16,
    use_local_band: bool = False,     # ★ 追加
    h_band_half_width: float = 5.0,   # ★ 追加
):
    """
    1つの衛星について:
      - 条件で df_sel を抽出
      - wavelet -> P(e,h)
      - R(e) を計算
      - 中央時刻に対応する1点 R_mid を返す（パス両端は使わない）

    戻り値 (dict):
      ok=False のときはスキップ理由を reason に入れる。
    """
    # direction は無視して、time_range + az_range のみで絞る
    df_sel, sig_col = select_observations(
        snr_table,
        doy=doy,
        sat=sat,
        signal=signal,
        az_range=az_range,
        direction=None,
        time_range_hour=time_range_hour,
        check_mode=False,
    )

    if len(df_sel) < min_points:
        return {"ok": False, "reason": "too_few_points", "sat": sat, "doy": doy}

    e = df_sel["elev_deg"].to_numpy()
    snr = df_sel[sig_col].to_numpy()

    # wavelet
    try:
        e_grid, h_grid_used, P = compute_wavelet_power_e_h(
            e,
            snr,
            h_grid,
            wavelength_m=wavelength_m,
            wavelet_name=wavelet_name,
            nyquist_strict=nyquist_strict,
        )
    except ValueError as exc:
        return {"ok": False, "reason": str(exc), "sat": sat, "doy": doy}

    # R(e)
    res = compute_R_from_power(e_grid, h_grid_used, P, h0, use_local_band=use_local_band, h_band_half_width=h_band_half_width)

    # --- ここで e 積分の比率を取り出す ---
    R_int = float(res["R_int"])
    R_int_dB = float(10.0 * np.log10(R_int)) if R_int > 0 else np.nan
    # -----------------------------------

    # 中央時刻（sec_of_day）の行を取る
    df_time = df_sel.sort_values("sec_of_day")
    idx_mid = len(df_time) // 2
    row_mid = df_time.iloc[idx_mid]
    sec_mid = float(row_mid["sec_of_day"])
    e_mid = float(row_mid["elev_deg"])
    az_mid = float(row_mid["az_deg"])

    # e_mid に最も近い R を代表値とする
    e_R = np.asarray(res["e"])
    R_R = np.asarray(res["R"])
    idx_R = int(np.argmin(np.abs(e_R - e_mid)))
    R_mid = float(R_R[idx_R])
    R_mid_dB = float(10.0 * np.log10(R_mid)) if R_mid > 0 else np.nan

    # パス全体の time/elev 範囲（チェック用に保存）
    t_start_hour = float(df_time["sec_of_day"].min() / 3600.0)
    t_end_hour = float(df_time["sec_of_day"].max() / 3600.0)
    e_start_deg = float(df_time["elev_deg"].min())
    e_end_deg = float(df_time["elev_deg"].max())

    return {
        "ok": True,
        "sat": sat,
        "doy": doy,
        "signal": signal,
        "df_sel": df_sel,
        "e_grid": res["e"],
        "R_grid": res["R"],
        "e_mid": e_mid,
        "az_mid": az_mid,
        "R_mid": R_mid,
        "R_mid_dB": R_mid_dB,
        "R_int": R_int,
        "R_int_dB": R_int_dB,
        "t_start_hour": t_start_hour,
        "t_end_hour": t_end_hour,
        "e_start_deg": e_start_deg,
        "e_end_deg": e_end_deg,
        "h0": h0,
    }


# Azimuth範囲フィルタ
def normalize_angle_deg(angle: float) -> float:
    """角度を 0〜360° に正規化。"""
    return float(np.mod(angle, 360.0))


def azimuth_in_range(az: pd.Series, az_min: float, az_max: float) -> pd.Series:
    """
    az_min〜az_max の範囲に入るかどうかを返す。
    - az は 0〜360° を想定
    - az_min, az_max は負でも 360 超でも OK（mod 360 で正規化）
    - 範囲が 360° をまたぐ場合（例: 300〜60°）も OR 条件で扱う
    """
    az_mod = np.mod(az.to_numpy(), 360.0)

    amin = normalize_angle_deg(az_min)
    amax = normalize_angle_deg(az_max)

    if amin <= amax:
        mask = (az_mod >= amin) & (az_mod <= amax)
    else:
        # 例: amin=290, amax=140 のとき
        # 290〜360° または 0〜140°
        mask = (az_mod >= amin) | (az_mod <= amax)

    return pd.Series(mask, index=az.index)

def add_direction_by_elev(df: pd.DataFrame, eps: float = 1e-3) -> pd.DataFrame:
    """
    sec_of_day の順に並べ、仰角の差分から昇り/下りを判定して
    'direction_flag' カラムに 'up' / 'down' / 'flat' を入れる。
    """
    if df.empty:
        df["direction_flag"] = []
        return df

    df = df.sort_values("sec_of_day").copy()

    de = df["elev_deg"].diff()  # 1 ステップ前との差
    # 先頭 NaN は 0 として処理
    de.iloc[0] = de.iloc[1] if len(de) > 1 else 0.0

    dir_flag = np.where(
        de > eps, "up",
        np.where(de < -eps, "down", "flat")
    )

    df["direction_flag"] = dir_flag
    return df


def select_observations(
    snr_table: pd.DataFrame,
    doy: int | None = None,
    sat: int | None = None,
    signal: str = "L1",
    az_range: tuple[float, float] | None = None,
    direction: str | None = None,
    time_range_hour: tuple[float, float] | None = None,
    check_mode: bool = False,
):
    """
    GNSS-IR 解析に使う観測点をフィルタする。

    引数:
        snr_table : main()/load_snr_directory() で作ったテーブル
        doy       : day of year。None なら全 DOY。
        sat       : 衛星番号。None なら全衛星。
        signal    : "L1", "L2", ... どの信号を使うか。
        az_range  : (az_min, az_max) [deg]。負/360超 OK。
        direction : None / "up" / "down"
                    → 仰角の変化から判定する。
        time_range_hour :
            (t_start, t_end) [hour]。
            例: (10.0, 10.5) → 10:00〜10:30。
            指定する場合は doy も指定しておくことを想定。
        check_mode :
            True のとき、衛星ごとのサマリ情報も返す。

    戻り値:
        check_mode=False のとき:
            df_sel, signal_col
        check_mode=True のとき:
            df_sel, signal_col, summary_df
              summary_df の列:
                ["sat", "direction_flag",
                 "time_start_hour", "time_end_hour",
                 "elev_min_deg", "elev_max_deg",
                 "available_signals", "n_points"]
    """
    if signal not in SIGNAL_COL_MAP:
        raise ValueError(
            f"未知の signal: {signal}. {list(SIGNAL_COL_MAP.keys())} から選んでください。"
        )

    signal_col = SIGNAL_COL_MAP[signal]

    df = snr_table.copy()

    # DOY で絞る
    if doy is not None:
        df = df[df["doy"] == doy]

    # 時間帯指定（hour -> sec_of_day）
    if time_range_hour is not None:
        if doy is None:
            raise ValueError("time_range_hour を使う場合は doy も指定してください。")
        t_start_h, t_end_h = time_range_hour
        t_start_s = t_start_h * 3600.0
        t_end_s = t_end_h * 3600.0
        df = df[(df["sec_of_day"] >= t_start_s) & (df["sec_of_day"] <= t_end_s)]

    # 衛星番号で絞る
    if sat is not None:
        df = df[df["sat"] == sat]

    # Azimuth 範囲で絞る
    if az_range is not None:
        az_min, az_max = az_range
        mask_az = azimuth_in_range(df["az_deg"], az_min, az_max)
        df = df[mask_az]

    # direction_flag を仰角の時間変化から付与
    # （direction=None のときも check_mode 用に付けておく）
    df = add_direction_by_elev(df)

    # up/down で絞る
    if direction == "up":
        df = df[df["direction_flag"] == "up"]
    elif direction == "down":
        df = df[df["direction_flag"] == "down"]

    # 最終的には仰角順に並べておく
    df = df.sort_values("elev_deg").reset_index(drop=True)

    # check_mode でサマリを作る
    if check_mode:
        # 衛星 × direction_flag ごとにまとめる
        groups = df.groupby(["sat", "direction_flag"], dropna=True)

        summary_rows = []
        for (sat_i, dir_flag), g in groups:
            if g.empty:
                continue

            # 時刻レンジ [hour]
            t_start = g["sec_of_day"].min() / 3600.0
            t_end = g["sec_of_day"].max() / 3600.0

            # 仰角レンジ [deg]
            e_min = g["elev_deg"].min()
            e_max = g["elev_deg"].max()

            # 利用可能な信号種類（L1/L2/L5/...）
            available_signals = []
            for sig_name, col_name in SIGNAL_COL_MAP.items():
                if col_name in g.columns and g[col_name].notna().any():
                    available_signals.append(sig_name)
            available_signals_str = ",".join(available_signals) if available_signals else ""

            summary_rows.append(
                {
                    "sat": sat_i,
                    "direction_flag": dir_flag,
                    "time_start_hour": t_start,
                    "time_end_hour": t_end,
                    "elev_min_deg": e_min,
                    "elev_max_deg": e_max,
                    "available_signals": available_signals_str,
                    "n_points": len(g),
                }
            )

        summary_df = pd.DataFrame(summary_rows).sort_values(
            ["sat", "direction_flag", "time_start_hour"]
        )

        return df, signal_col, summary_df

    # 通常モード
    return df, signal_col

import os

def batch_R_all_sats(
    snr_table: pd.DataFrame,
    doy: int,
    signal: str,
    az_range: tuple[float, float],
    time_range_hour: tuple[float, float],
    h_grid: np.ndarray,
    h0: float,
    out_dir: str = "./R_results",
    wavelength_m: float = LAMBDA_L1,
    wavelet_name: str = "cmor5.0-1.0",
    nyquist_strict: bool = True,
    min_points: int = 16,
    use_local_band: bool = False,
    h_band_half_width: float = 5.0,
):
    """
    指定した doy, signal, az_range, time_range_hour について、
    データのある全衛星を対象に R を計算し、
    各衛星パスの「中央時刻」の 1 点をまとめて .res に書き出す。

    出力列:
      doy, sat, sig_code,
      az_mid_deg, e_mid_deg,
      R_mid_lin, R_mid_dB,
      R_int_lin, R_int_dB,
      t_start_hour, t_end_hour,
      e_start_deg, e_end_deg,
      h0_m
    """
    os.makedirs(out_dir, exist_ok=True)

    sats = sorted(snr_table[snr_table["doy"] == doy]["sat"].unique())
    summary_rows = []

    for sat in sats:
        res_sat = compute_R_for_single_sat(
            snr_table=snr_table,
            doy=doy,
            sat=sat,
            signal=signal,
            az_range=az_range,
            time_range_hour=time_range_hour,
            h_grid=h_grid,
            h0=h0,
            wavelength_m=wavelength_m,
            wavelet_name=wavelet_name,
            nyquist_strict=nyquist_strict,
            min_points=min_points,
            use_local_band=use_local_band,
            h_band_half_width=h_band_half_width,
        )

        if not res_sat.get("ok", False):
            # 必要ならここでログ出力
            continue

        sig_code = SIGNAL_CODE_MAP.get(signal, -1)

        summary_rows.append([
            float(doy),
            float(sat),
            float(sig_code),
            res_sat["az_mid"],
            res_sat["e_mid"],
            res_sat["R_mid"],       # 真の R_mid
            res_sat["R_mid_dB"],
            res_sat["R_int"],       # e 積分した R
            res_sat["R_int_dB"],
            res_sat["t_start_hour"],
            res_sat["t_end_hour"],
            res_sat["e_start_deg"],
            res_sat["e_end_deg"],
            float(h0),
        ])

    if not summary_rows:
        print("有効な衛星がありませんでした。")
        return None

    summary_arr = np.array(summary_rows, dtype=float)

    out_path = os.path.join(out_dir, f"R_summary_doy{doy}_{signal}.res")
    header = (
        "# R summary for all satellites\n"
        f"# doy={doy}, signal={signal} (sig_code map: {SIGNAL_CODE_MAP})\n"
        "# columns:\n"
        "# doy, sat, sig_code, az_mid_deg, e_mid_deg,\n"
        "# R_mid_lin, R_mid_dB, R_int_lin, R_int_dB,\n"
        "# t_start_hour, t_end_hour,\n"
        "# e_start_deg, e_end_deg, h0_m\n"
    )

    fmt = [
        "%3d",    # doy
        "%3d",    # sat
        "%2d",    # sig_code
        "%8.3f",  # az_mid_deg
        "%8.3f",  # e_mid_deg
        "%10.6f", # R_mid_lin
        "%8.3f",  # R_mid_dB
        "%10.6f", # R_int_lin
        "%8.3f",  # R_int_dB
        "%7.3f",  # t_start_hour
        "%7.3f",  # t_end_hour
        "%8.3f",  # e_start_deg
        "%8.3f",  # e_end_deg
        "%6.2f",  # h0_m
    ]

    np.savetxt(out_path, summary_arr, header=header, fmt=fmt)
    print(f"summary written to {out_path}")

    return summary_arr


# 1. SNR 読み込み
snr_dict, snr_table = main(2020, "calc")
print("読み込み完了")

# 1.5 条件に合う観測があるかのチェック
df_sel_check, sig_col_check, summary = select_observations(
    snr_table,
    doy=281,
    sat=None,
    signal="L1",
    az_range=(-140, 70),
    direction=None,
    time_range_hour=(0.5, 23.5),
    check_mode=True,
)
display(summary)

# 2. 条件に合う観測を抽出（実際に解析する1衛星）
doy = 281
sat = 1
signal = "L1"
az_range = (-140, 70)
direction = "up"
time_range_hour = (0.5, 23.5)

df_sel, sig_col = select_observations(
    snr_table,
    doy=doy,
    sat=sat,
    signal=signal,
    az_range=az_range,
    direction=direction,
    time_range_hour=time_range_hour,
    check_mode=False,
)

e = df_sel["elev_deg"].to_numpy()
snr = df_sel[sig_col].to_numpy()

# 3. ウェーブレットから P(e,h) を作る
h_grid = np.linspace(0.5, 20.0, 196)  # 解析したい高さレンジ

try:
    e_grid, h_grid_used, P = compute_wavelet_power_e_h(e, snr, h_grid)
except ValueError as exc:
    print(f"skip sat={sat}, sig={signal}: {exc}")
else:
    # プロット
    plot_wavelet_power_e_h(
        e_grid,
        h_grid_used,
        P,
        sat=sat,
        signal=signal,
        doy=doy,
        az_range=az_range,
        direction=direction,
    )

    # 4. 真の RH h0 を指定して R(e) を計算
    h0 = 12.5
    res = compute_R_from_power(e_grid, h_grid_used, P, h0, coh_h_range=(9.0, 20.0), inc_h_range=(0.5, 5.0))

    # 5. R(e) プロット
    plot_R_e(
        res["e"],
        res["R"],
        sat=sat,
        signal=signal,
        doy=doy,
        az_range=az_range,
        direction=direction,
        h0=h0,
    )

    # 6. resファイルに書き出し（e, R, az）
    save_R_result_resfile(
        filepath=f"R_doy{doy}_sat{sat}_{signal}.res",
        doy=doy,
        sat=sat,
        signal=signal,
        df_sel=df_sel,
        res=res,
    )

# 7. 全衛星で R を計算してまとめる
doy = 286
signal = "L1"
az_range = (-140, 70)
time_range_hour = (0.5, 23.5)
h_grid = np.linspace(0.5, 20.0, 196)
h0 = 12.5

summary = batch_R_all_sats(
    snr_table=snr_table,
    doy=doy,
    signal=signal,
    az_range=az_range,
    time_range_hour=time_range_hour,
    h_grid=h_grid,
    h0=h0,
    out_dir="./R_results",
    use_local_band=True,       # ★ ここを True に
    h_band_half_width=5.0,     # ★ RH ± 5 m 範囲だけ使用
)




読み込み完了


,sat,direction_flag,time_start_hour,time_end_hour,elev_min_deg,elev_max_deg,available_signals,n_points
0,1,down,0.500000,23.500000,4.8159,43.4155,"L1,L2,L5,L6",481
1,1,up,7.666667,23.458333,6.3304,43.4873,"L1,L2,L5,L6",120
2,2,down,9.833333,15.000000,25.5300,60.5585,"L1,L2,L5,L6",223
3,2,up,9.841667,15.008333,25.7363,64.7767,"L1,L2,L5,L6",399
4,3,down,0.516667,23.475000,5.0017,56.5913,"L1,L2,L5,L6",314
...,...,...,...,...,...,...,...,...
62,30,up,3.700000,7.325000,6.6960,87.4543,"L1,L2,L5,L6",239
63,31,down,0.508333,2.200000,12.1934,42.3462,"L1,L2,L5,L6",124
64,31,up,0.500000,23.500000,16.7409,84.5116,"L1,L2,L5,L6",233
65,32,down,18.025000,23.425000,12.4790,60.7897,"L1,L2,L5,L6",317


skip sat=1, sig=L1: 指定された h_max=20.000 m は このデータのナイキスト限界 h_max≈17.404 m を超えています。
summary written to ./R_results/R_summary_doy286_L1.res


In [41]:
# カラム名一覧（= dict のキー）
col_names = list(res.keys())
print(col_names)
# -> ['e', 'P_coh', 'P_total', 'P_inc', 'R', 'sigma_coh']

# 行数（= 各配列の要素数。どれも同じ長さのはずなので e で代表）
n_rows = len(res["e"])
print(n_rows)


['e', 'P_coh', 'P_total', 'P_inc', 'R', 'sigma_coh']
386
